# M4 · 快计算：注意力、量化与显存

**核心问题：** 更快的注意力内核与 INT8 量化，哪一种更有工程价值？对照相同 B0 40步基线，区分吞吐、显存占用与画质；不要将各自加速比直接相乘。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M04_faster_computation/LESSON.zh-CN.md](../course/M04_faster_computation/LESSON.zh-CN.md)。切换语言：[English](./04_faster_computation.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
rows = ns.summary_rows()
base = rows["B0"]
for arm in ("F1", "F2"):
    v = rows[arm]
    gain = base["warm_total_median_s"] / v["warm_total_median_s"]
    delta_mb = base["vram_device_peak_mb"] - v["vram_device_peak_mb"]
    print(f"{arm}: speedup={gain:.3f}x, device peak saved={delta_mb:.1f} MB")
plot_values = {name: rows[name]["warm_total_median_s"] for name in ("B0", "F1", "F2")}
plot_ylabel, plot_title = "warm seconds", "Same 40-step workload"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
quality = ns.read_json("evidence/quality/F2/quality.json")
assert quality["semantics"] == "fidelity"
print("F2 LPIPS semantics:", quality["semantics"], "(distance from B0 numerics)")
for p, value in quality["median"].items():
    print(p, f"LPIPS median={value:.4f}")
print("LPIPS is NOT a general image-quality score and does not replace human review.")


## 可选 GPU 真机冒烟——必须显式授权

需要 ROCm 环境、模型与 GPU 额度。CI 不执行；结果只存于 `runs/`，不得混入官方证据。

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("F2", "notebook-f2", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## 学生思考 / 自检

将节约的显存同时换算为十进制 GB 和二进制 GiB，并说明为何不能混用单位。为什么量化只带来约 1.08×，却能明显降低显存？

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
